# U11 Task 3 — 주동작 원판 identity 추적 평가

U11 Task 2가 자동으로 고른 원판이 영상 끝까지 같은 원판으로 연결되는지 검증합니다. 새 모델을 학습하거나 추론하지 않고 저장된 후보·track을 재사용합니다. 원판이 있는 8영상에서 시간축 5프레임씩 최대 40개 bbox만 라벨링하고, B05는 `N/A: 원판 없음`으로 유지합니다.

핵심 지표는 `track_correct_rate`, `wrong_target_rate`, 누락률, 최장 누락, 평균 IoU와 정규화 중심 오차입니다. `track coverage`만으로 정확하다고 결론 내리지 않습니다.

In [ ]:
from pathlib import Path
import math
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from snatch_analysis.pipeline import save_json_atomically
from snatch_analysis.plate import (
    aggregate_plate_evaluations,
    cancel_plate_label,
    evaluate_plate_video,
    save_yolo_plate_label,
    yolo_bbox_to_opencv,
)
from snatch_analysis.video import iter_video_frames

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요.')
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

## 1. 설정

처음에는 모든 실행 플래그를 `False`로 둡니다. 라벨링할 때만 `RUN_U11_TASK3_LABELING=True`, 라벨 검수 그림을 만들 때만 `RUN_U11_TASK3_REVIEW=True`, 40개 bbox를 모두 확인한 뒤에만 `RUN_U11_TASK3_EVALUATION=True`로 바꿉니다.

In [ ]:
U11_TASK3_VIDEO_IDS = (
    'a02_sn_f_01', 'a06_sn_h_01', 'a08_sn_f_01',
    'b02_sn_f_01', 'b04_sn_f_01', 'b05_sn_f_01',
    'c03_sn_f_01', 'c04_sn_f_01', 'c05_sn_f_01',
)
U11_TASK3_SAMPLE_COUNT = 5
U11_TASK3_IOU_THRESHOLD = 0.50
U11_TASK3_ANNOTATOR = ''
RUN_U11_TASK3_LABELING = False
OVERWRITE_U11_TASK3_LABELS = False
RUN_U11_TASK3_REVIEW = False
RUN_U11_TASK3_EVALUATION = False

U11_TASK3_ROOT = PROJECT_ROOT / 'artifacts/u11_primary_plate_selection/task3'
U11_TASK3_INPUT_ROOT = PROJECT_ROOT / 'artifacts/u11_primary_plate_selection/task2/inputs'
U11_TASK2_TARGET_PATH = PROJECT_ROOT / 'artifacts/u11_primary_plate_selection/task2/manual_targets.csv'
U11_TASK3_LABEL_ROOT = PROJECT_ROOT / 'data/annotations/u11_task3_plate_identity_labels'
U11_TASK3_MANIFEST_PATH = PROJECT_ROOT / 'data/metadata/videos.csv'
U11_TASK3_SAMPLE_PATH = U11_TASK3_ROOT / 'sample_frames.csv'
U11_TASK3_FRAME_EVALUATION_PATH = U11_TASK3_ROOT / 'frame_evaluation.csv'
U11_TASK3_VIDEO_SUMMARY_PATH = U11_TASK3_ROOT / 'per_video_summary.csv'
U11_TASK3_GROUP_SUMMARY_PATH = U11_TASK3_ROOT / 'group_summary.csv'
U11_TASK3_SUMMARY_PATH = U11_TASK3_ROOT / 'run_summary.json'

## 2. 평가 영상과 희소 프레임

각 영상의 0%, 25%, 50%, 75%, 100% 지점에 가까운 프레임을 고릅니다. 아직 9영상에 phase 이벤트가 없으므로 수동 phase를 새로 만들지 않고 시간축만 균등하게 봅니다.

In [ ]:
manifest = pd.read_csv(U11_TASK3_MANIFEST_PATH)
if manifest['video_id'].duplicated().any():
    raise ValueError('Manifest video_id가 중복되었습니다.')
metadata_by_video = manifest.set_index('video_id', drop=False)
missing_video_ids = sorted(set(U11_TASK3_VIDEO_IDS) - set(metadata_by_video.index.astype(str)))
if missing_video_ids:
    raise ValueError('Manifest에 없는 영상: ' + ', '.join(missing_video_ids))
task3_rows = metadata_by_video.loc[list(U11_TASK3_VIDEO_IDS)].copy()
if task3_rows['read_status'].astype(str).ne('ok').any():
    raise ValueError('U11 Task 3 영상의 read_status는 모두 ok여야 합니다.')
if task3_rows['frame_count'].astype(int).lt(U11_TASK3_SAMPLE_COUNT).any():
    raise ValueError('sample_count보다 짧은 영상이 있습니다.')

task2_targets = pd.read_csv(U11_TASK2_TARGET_PATH)
if set(task2_targets['video_id'].astype(str)) != set(U11_TASK3_VIDEO_IDS):
    raise ValueError('U11 Task 2 수동 target 영상과 Task 3 영상이 다릅니다.')
target_status_by_video = task2_targets.set_index('video_id')['target_status'].astype(str)

sample_rows = []
for video_id, metadata in task3_rows.iterrows():
    frame_count = int(metadata['frame_count'])
    frame_indices = np.linspace(
        0, frame_count - 1, num=U11_TASK3_SAMPLE_COUNT, dtype=int
    )
    if len(set(frame_indices.tolist())) != U11_TASK3_SAMPLE_COUNT:
        raise ValueError(f'{video_id} sample frame이 중복되었습니다.')
    plate_present = (
        'no' if target_status_by_video.loc[video_id] == 'no_plate' else 'yes'
    )
    for sample_order, frame_index in enumerate(frame_indices, start=1):
        sample_rows.append({
            'video_id': video_id,
            'group': str(metadata['group']),
            'frame_index': int(frame_index),
            'sample_order': sample_order,
            'plate_present': plate_present,
            'label_filename': f'{video_id}_frame_{int(frame_index):04d}.txt',
        })
sample_frames = pd.DataFrame(sample_rows)
if len(sample_frames) != len(U11_TASK3_VIDEO_IDS) * U11_TASK3_SAMPLE_COUNT:
    raise RuntimeError('U11 Task 3 sample 수가 예상과 다릅니다.')
U11_TASK3_ROOT.mkdir(parents=True, exist_ok=True)
sample_frames.to_csv(U11_TASK3_SAMPLE_PATH, index=False)
display(sample_frames)

def load_selected_frames(rows):
    frames = {}
    for video_id, video_rows in rows.groupby('video_id', sort=False):
        wanted = set(video_rows['frame_index'].astype(int))
        if not wanted:
            continue
        last_wanted = max(wanted)
        metadata = task3_rows.loc[video_id]
        for frame_index, _, frame in iter_video_frames(
            PROJECT_ROOT / str(metadata['path'])
        ):
            if frame_index in wanted:
                frames[(video_id, frame_index)] = frame.copy()
            if frame_index >= last_wanted:
                break
        missing = wanted - {
            frame_index for stored_video_id, frame_index in frames
            if stored_video_id == video_id
        }
        if missing:
            raise RuntimeError(f'{video_id} 프레임을 읽지 못했습니다: {sorted(missing)}')
    return frames

## 3. 최대 40개 bbox 수동 라벨링

`U11_TASK3_ANNOTATOR`를 입력하고 `RUN_U11_TASK3_LABELING=True`로 실행합니다. OpenCV 창에서 주동작자가 드는 카메라 쪽 원판 하나만 지정합니다. 이미 저장된 라벨은 건너뛰므로 중간에 멈춰도 이어서 할 수 있습니다. 잘못된 라벨만 다시 그릴 때 `OVERWRITE_U11_TASK3_LABELS=True`를 사용합니다. B05의 5프레임은 원판 없음으로 자동 제외됩니다.

In [ ]:
label_actions = []
if not RUN_U11_TASK3_LABELING:
    print('Task 3 라벨링 대기: RUN_U11_TASK3_LABELING=True로 바꾸세요.')
else:
    if not U11_TASK3_ANNOTATOR.strip():
        raise ValueError('U11_TASK3_ANNOTATOR를 입력하세요.')
    U11_TASK3_LABEL_ROOT.mkdir(parents=True, exist_ok=True)
    for _, video_samples in sample_frames.groupby('video_id', sort=False):
        needs_label = video_samples.loc[video_samples['plate_present'].eq('yes')]
        if not OVERWRITE_U11_TASK3_LABELS:
            needs_label = needs_label.loc[
                ~needs_label['label_filename'].map(
                    lambda name: (U11_TASK3_LABEL_ROOT / name).exists()
                )
            ]
        video_frames = load_selected_frames(needs_label)
        for sample in video_samples.itertuples(index=False):
            if sample.plate_present == 'no':
                label_actions.append({
                    'video_id': sample.video_id, 'frame_index': sample.frame_index,
                    'action': 'not_applicable_no_plate',
                })
                continue
            label_path = U11_TASK3_LABEL_ROOT / sample.label_filename
            if label_path.exists() and not OVERWRITE_U11_TASK3_LABELS:
                label_actions.append({
                    'video_id': sample.video_id, 'frame_index': sample.frame_index,
                    'action': 'skipped',
                })
                continue
            metadata = task3_rows.loc[sample.video_id]
            frame = video_frames[(sample.video_id, int(sample.frame_index))]
            bbox = cv2.selectROI(
                f'{sample.video_id} frame={sample.frame_index}',
                frame, showCrosshair=True, fromCenter=False,
            )
            cv2.destroyAllWindows()
            if bbox[2] <= 0 or bbox[3] <= 0:
                action = cancel_plate_label(
                    label_path, overwrite=OVERWRITE_U11_TASK3_LABELS
                )
            else:
                action = save_yolo_plate_label(
                    label_path, tuple(float(value) for value in bbox),
                    image_width=int(metadata['width']),
                    image_height=int(metadata['height']),
                    overwrite=OVERWRITE_U11_TASK3_LABELS,
                )
            label_actions.append({
                'video_id': sample.video_id, 'frame_index': sample.frame_index,
                'action': action,
            })
    display(pd.DataFrame(label_actions))

## 4. 라벨 완성도와 contact sheet

완료 수를 먼저 확인합니다. `RUN_U11_TASK3_REVIEW=True`로 실행하면 영상별 5프레임을 한 페이지에 초록 bbox로 저장합니다. 40개 bbox가 모두 주동작 원판인지 확인한 뒤 평가로 넘어갑니다.

In [ ]:
label_progress = sample_frames.copy()
label_progress['label_status'] = [
    'not_applicable' if row.plate_present == 'no'
    else 'complete' if (U11_TASK3_LABEL_ROOT / row.label_filename).exists()
    else 'pending'
    for row in label_progress.itertuples(index=False)
]
required_label_count = int(label_progress['plate_present'].eq('yes').sum())
completed_label_count = int(label_progress['label_status'].eq('complete').sum())
print(f'원판 bbox 완료: {completed_label_count}/{required_label_count}')
display(label_progress.groupby(['group', 'label_status']).size().rename('count').reset_index())

if RUN_U11_TASK3_REVIEW:
    completed_samples = label_progress.loc[
        label_progress['label_status'].isin(['complete', 'not_applicable'])
    ].reset_index(drop=True)
    for page_index, (_, page) in enumerate(
        completed_samples.groupby('video_id', sort=False), start=1
    ):
        page_frames = load_selected_frames(page)
        figure, axes = plt.subplots(3, 3, figsize=(15, 21), squeeze=False)
        for axis in axes.flat:
            axis.axis('off')
        for axis, sample in zip(axes.flat, page.itertuples(index=False)):
            metadata = task3_rows.loc[sample.video_id]
            frame = page_frames[(sample.video_id, int(sample.frame_index))].copy()
            if sample.label_status == 'complete':
                tokens = (U11_TASK3_LABEL_ROOT / sample.label_filename).read_text(
                    encoding='utf-8'
                ).strip().split()
                if len(tokens) != 5 or tokens[0] != '0':
                    raise ValueError(f'잘못된 원판 라벨: {sample.label_filename}')
                bbox = yolo_bbox_to_opencv(
                    tuple(float(value) for value in tokens[1:]),
                    image_width=int(metadata['width']),
                    image_height=int(metadata['height']),
                )
                x, y, width, height = (int(round(value)) for value in bbox)
                cv2.rectangle(frame, (x, y), (x + width, y + height), (0, 255, 0), 4)
            axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            axis.set_title(
                f'{sample.video_id} | frame={sample.frame_index} | {sample.label_status}'
            )
            axis.axis('off')
        figure.tight_layout()
        output_path = U11_TASK3_ROOT / f'label_review_{page_index:02d}.png'
        figure.savefig(output_path, dpi=120, bbox_inches='tight')
        plt.show()
        plt.close(figure)
        print(f'라벨 검수 그림: {output_path}')

## 5. 동일 원판 identity 평가

40개 bbox 검수를 마친 뒤 `RUN_U11_TASK3_EVALUATION=True`로 실행합니다. 후보 정확도와 track 정확도를 분리하고, 좌표 반환률이 높아도 다른 원판이면 `wrong_target`으로 기록합니다.

In [ ]:
task3_frame_evaluation = pd.DataFrame()
task3_video_summary = pd.DataFrame()
task3_group_summary = pd.DataFrame()
if not RUN_U11_TASK3_EVALUATION:
    print('Task 3 평가 대기: bbox 40개 검수 후 RUN_U11_TASK3_EVALUATION=True로 바꾸세요.')
else:
    if not U11_TASK3_ANNOTATOR.strip():
        raise ValueError('U11_TASK3_ANNOTATOR를 입력하세요.')
    pending = label_progress.loc[label_progress['label_status'].eq('pending')]
    if not pending.empty:
        raise ValueError(f'원판 bbox 라벨이 {len(pending)}개 부족합니다.')

    frame_evaluations = []
    video_summaries = []
    for video_id, metadata in task3_rows.iterrows():
        video_samples = sample_frames.loc[sample_frames['video_id'].eq(video_id)]
        truth_rows = []
        for sample in video_samples.itertuples(index=False):
            if sample.plate_present == 'no':
                bbox = (math.nan, math.nan, math.nan, math.nan)
            else:
                tokens = (U11_TASK3_LABEL_ROOT / sample.label_filename).read_text(
                    encoding='utf-8'
                ).strip().split()
                if len(tokens) != 5 or tokens[0] != '0':
                    raise ValueError(f'잘못된 원판 라벨: {sample.label_filename}')
                bbox = yolo_bbox_to_opencv(
                    tuple(float(value) for value in tokens[1:]),
                    image_width=int(metadata['width']),
                    image_height=int(metadata['height']),
                )
            truth_rows.append({
                'video_id': video_id, 'frame_index': int(sample.frame_index),
                'plate_present': sample.plate_present,
                'plate_bbox_x': bbox[0], 'plate_bbox_y': bbox[1],
                'plate_bbox_width': bbox[2], 'plate_bbox_height': bbox[3],
            })
        ground_truth = pd.DataFrame(truth_rows)
        input_root = U11_TASK3_INPUT_ROOT / video_id
        candidates = pd.read_csv(input_root / 'plate_candidates.csv')
        track_path = input_root / 'plate_track.csv'
        track = pd.read_csv(track_path) if track_path.exists() else None
        frame_evaluation, video_summary = evaluate_plate_video(
            ground_truth, candidates, track,
            frame_count=int(metadata['frame_count']), fps=float(metadata['fps']),
            iou_threshold=U11_TASK3_IOU_THRESHOLD,
            missing_track_reason='initial_plate_not_available',
        )
        video_summary['group'] = str(metadata['group'])
        frame_evaluations.append(frame_evaluation)
        video_summaries.append(video_summary)

    task3_frame_evaluation = pd.concat(frame_evaluations, ignore_index=True)
    task3_video_summary = pd.DataFrame(video_summaries)
    overall = aggregate_plate_evaluations(task3_video_summary)
    group_summaries = []
    for group, rows in task3_video_summary.groupby('group', sort=True):
        group_summary = aggregate_plate_evaluations(rows)
        group_summaries.append({'group': group, **group_summary})
    task3_group_summary = pd.DataFrame(group_summaries)

    detected_iou = pd.to_numeric(
        task3_frame_evaluation['track_iou'], errors='coerce'
    )
    expected_ground_truth_count = required_label_count
    evidence_complete = (
        overall['ground_truth_frame_count'] == expected_ground_truth_count
        and task3_group_summary['ground_truth_frame_count'].gt(0).all()
    )
    decision = (
        'pass'
        if evidence_complete
        and overall['wrong_target_count'] == 0
        and overall['track_correct_count'] == expected_ground_truth_count
        else 'hold' if evidence_complete else 'insufficient_evidence'
    )
    task3_summary = {
        'status': 'complete', 'decision': decision,
        'video_count': len(U11_TASK3_VIDEO_IDS),
        'plate_video_count': int((target_status_by_video != 'no_plate').sum()),
        'no_plate_video_count': int((target_status_by_video == 'no_plate').sum()),
        'sample_count_per_video': U11_TASK3_SAMPLE_COUNT,
        'iou_threshold': U11_TASK3_IOU_THRESHOLD,
        'annotator': U11_TASK3_ANNOTATOR.strip(),
        **overall,
        'mean_track_iou': float(detected_iou.mean()) if detected_iou.notna().any() else None,
    }
    U11_TASK3_ROOT.mkdir(parents=True, exist_ok=True)
    task3_frame_evaluation.to_csv(U11_TASK3_FRAME_EVALUATION_PATH, index=False)
    task3_video_summary.to_csv(U11_TASK3_VIDEO_SUMMARY_PATH, index=False)
    task3_group_summary.to_csv(U11_TASK3_GROUP_SUMMARY_PATH, index=False)
    save_json_atomically(task3_summary, U11_TASK3_SUMMARY_PATH)
    display(task3_frame_evaluation)
    display(task3_video_summary)
    display(task3_group_summary)
    display(pd.DataFrame([task3_summary]))

## 6. 실패 프레임만 검토

평가 후 `track_outcome != correct`인 원판 프레임만 보여줍니다. 초록은 수동 정답, 빨강은 tracker입니다. 실패가 없으면 새로운 tracker 코드를 추가하지 않고 현재 방법을 유지합니다.

In [ ]:
if task3_frame_evaluation.empty and U11_TASK3_FRAME_EVALUATION_PATH.exists():
    task3_frame_evaluation = pd.read_csv(U11_TASK3_FRAME_EVALUATION_PATH)
if task3_frame_evaluation.empty:
    print('Task 3 실패 검토 대기: 먼저 평가를 실행하세요.')
else:
    failures = task3_frame_evaluation.loc[
        task3_frame_evaluation['plate_present'].eq('yes')
        & task3_frame_evaluation['track_outcome'].ne('correct')
    ].reset_index(drop=True)
    print(f'실패 frame: {len(failures)}')
    for page_index, (_, page) in enumerate(
        failures.groupby('video_id', sort=False), start=1
    ):
        page_frames = load_selected_frames(page)
        figure, axes = plt.subplots(3, 3, figsize=(15, 21), squeeze=False)
        for axis in axes.flat:
            axis.axis('off')
        for axis, failure in zip(axes.flat, page.itertuples(index=False)):
            frame = page_frames[(failure.video_id, int(failure.frame_index))].copy()
            truth_start = (
                int(round(failure.ground_truth_bbox_x1)),
                int(round(failure.ground_truth_bbox_y1)),
            )
            truth_end = (
                int(round(failure.ground_truth_bbox_x2)),
                int(round(failure.ground_truth_bbox_y2)),
            )
            cv2.rectangle(frame, truth_start, truth_end, (0, 255, 0), 4)
            if failure.track_status == 'ok':
                track_start = (
                    int(round(failure.track_bbox_x1)), int(round(failure.track_bbox_y1))
                )
                track_end = (
                    int(round(failure.track_bbox_x2)), int(round(failure.track_bbox_y2))
                )
                cv2.rectangle(frame, track_start, track_end, (0, 0, 255), 3)
            axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            axis.set_title(
                f'{failure.video_id} | frame={failure.frame_index} | {failure.track_outcome}'
            )
            axis.axis('off')
        figure.tight_layout()
        output_path = U11_TASK3_ROOT / f'failure_review_{page_index:02d}.png'
        figure.savefig(output_path, dpi=120, bbox_inches='tight')
        plt.show()
        plt.close(figure)
        print(f'실패 검수 그림: {output_path}')

## 해석 기준

- `candidate_recall`: 수동 정답 위치에 detector 후보가 있었던 비율
- `track_correct_rate`: 수동 정답과 IoU 기준을 통과한 track 비율
- `wrong_target_rate`: track이 좌표는 냈지만 다른 원판을 선택한 비율
- `plate_ok_rate`: 전체 영상 프레임에서 좌표를 반환한 비율이며 정확도와 다름
- `longest_non_ok_gap_frames`: 가장 긴 연속 누락·확인 필요 구간
- `mean_track_iou`, `mean_normalized_center_error`: 맞는 위치에 얼마나 가깝게 놓였는지 보는 보조 지표

표본은 8영상 40프레임이므로 54개 전체 일반화를 주장하지 않습니다. 실패가 실제로 확인될 때만 continuity 규칙을 한 축씩 바꿉니다.